# Student 360 in 1 Hour - Part 1: Build the Student 360
⏱ **~20 min** (mostly waiting for cells to run). Icons: 🛠️ action, 📌 note.

**Scenario:** Lakeside University loses students and finds out too late. The signals exist in **six systems that each call a student something different** (SIS, LMS, Financial Aid, Bursar, Advising CRM, Campus card). All data is synthetic.

**What you will do:** create an isolated lab, load the six sources, resolve identity, engineer features with **dynamic tables**, clean advising notes with **Cortex AI functions**, and assemble one governed row per student.

**Snowflake features you touch:** roles and grants, dynamic tables (incremental), `AI_REDACT`, `AI_CLASSIFY`, data quality checks.

## 1. Lab foundation (isolated role, database, warehouse)
🛠️ Everything lives under `STUDENT360_LAB` so it can be dropped as a unit. Run the next cells in order.

In [ ]:
%%sql -r create_role
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS STUDENT360_LAB_ADMIN COMMENT = 'Student 360 workshop lab role';
GRANT ROLE STUDENT360_LAB_ADMIN TO ROLE ACCOUNTADMIN;
SET me = CURRENT_USER();
GRANT ROLE STUDENT360_LAB_ADMIN TO USER IDENTIFIER($me);
SELECT 'STUDENT360_LAB_ADMIN created and granted' AS STATUS;

In [ ]:
%%sql -r create_db
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS STUDENT360_LAB COMMENT = 'Student 360 + CoWork workshop (Higher Ed)';
CREATE SCHEMA IF NOT EXISTS STUDENT360_LAB.RAW        COMMENT = 'Simulated source systems';
CREATE SCHEMA IF NOT EXISTS STUDENT360_LAB.CURATED    COMMENT = 'Identity-resolved Student 360';
CREATE SCHEMA IF NOT EXISTS STUDENT360_LAB.GOVERNANCE COMMENT = 'Policies and access scope';
CREATE SCHEMA IF NOT EXISTS STUDENT360_LAB.SEMANTIC   COMMENT = 'Semantic views';
CREATE SCHEMA IF NOT EXISTS STUDENT360_LAB.AGENTS     COMMENT = 'Agents, search services, eval data';
GRANT OWNERSHIP ON ALL SCHEMAS IN DATABASE STUDENT360_LAB TO ROLE STUDENT360_LAB_ADMIN REVOKE CURRENT GRANTS;
GRANT OWNERSHIP ON DATABASE STUDENT360_LAB TO ROLE STUDENT360_LAB_ADMIN COPY CURRENT GRANTS;
SELECT 'Database and schemas ready' AS STATUS;

In [ ]:
%%sql -r create_wh_grants
USE ROLE ACCOUNTADMIN;
CREATE WAREHOUSE IF NOT EXISTS STUDENT360_LAB_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Student 360 workshop warehouse';
GRANT OWNERSHIP ON WAREHOUSE STUDENT360_LAB_WH TO ROLE STUDENT360_LAB_ADMIN COPY CURRENT GRANTS;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE STUDENT360_LAB_ADMIN;
GRANT CREATE ROLE ON ACCOUNT TO ROLE STUDENT360_LAB_ADMIN;
GRANT CREATE USER ON ACCOUNT TO ROLE STUDENT360_LAB_ADMIN;
GRANT CREATE SNOWFLAKE INTELLIGENCE ON ACCOUNT TO ROLE STUDENT360_LAB_ADMIN;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE STUDENT360_LAB_ADMIN;
SELECT 'Warehouse + grants ready' AS STATUS;

In [ ]:
%%sql -r cross_region
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

In [ ]:
%%sql -r set_context
USE ROLE STUDENT360_LAB_ADMIN;
USE DATABASE STUDENT360_LAB;
USE SCHEMA RAW;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_WAREHOUSE();

In [ ]:
%%sql -r sim_helper
CREATE OR REPLACE FUNCTION RAW.SIM_RISK(N NUMBER)
  RETURNS FLOAT
  COMMENT = 'SIMULATION ONLY - latent 0..1 churn propensity used to generate correlated synthetic data'
  AS $$ POWER((ABS(HASH(N, 'risk')) % 1000) / 1000.0, 1.8)::FLOAT $$;
SELECT RAW.SIM_RISK(1) AS example;

## 2. Load six source systems
🛠️ Each source has its **own student key format**: `S0000123`, login email, `FA-100007` plus a bare number, `STU-0000123`, card id. ~1,200 students. Deterministic, so every attendee gets identical data.

In [ ]:
%%sql -r load_sis_student
CREATE OR REPLACE TABLE RAW.SIS_STUDENT
COMMENT = 'Registrar SIS - one row per student. Key: STUDENT_ID (S + 7 digits)' AS
WITH g AS (SELECT SEQ4() + 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => 1200))),
b AS (
  SELECT n, RAW.SIM_RISK(n) AS r,
    GET(ARRAY_CONSTRUCT('Maria','James','Aisha','Wei','Carlos','Priya','Liam','Fatima','Noah','Sofia','Daniel','Mei','Omar','Emma','Jamal','Hannah','Diego','Yuki','Grace','Ethan','Olivia','Mateo','Amara','Lucas','Isabella','Kenji','Zara','Henry','Layla','Samuel','Chloe','Andre','Nia','Tyler','Elena','Rahul','Megan','Hassan','Lily','Victor','Anika','Jack','Camila','Ravi','Sienna','Marcus','Leila','Owen','Tessa','Bilal'), MOD(n, 50))::STRING AS fn,
    GET(ARRAY_CONSTRUCT('Garcia','Nguyen','Johnson','Patel','Kim','Hernandez','Smith','Okafor','Chen','Martinez','Brown','Singh','Lopez','Davis','Tanaka','Wilson','Ahmed','Rossi','Cohen','Mitchell','Thompson','Reyes','Abdi','Kowalski','Fernandez','Park','Walker','Hussain','Moreau','Bennett','Yamamoto','Castillo','Adeyemi','Foster','Ivanov','Murphy','Das','Silva','Young','Haddad'), MOD(FLOOR(n / 50), 40))::STRING AS ln,
    GET(ARRAY_CONSTRUCT('Engineering','Arts & Sciences','Business','Nursing & Health','Education'), MOD(ABS(HASH(n,'col')), 5))::STRING AS college,
    GET(ARRAY_CONSTRUCT('Freshman','Sophomore','Junior','Senior'), MOD(ABS(HASH(n,'cl')), 4))::STRING AS class_level
  FROM g
)
SELECT
  'S' || LPAD(n, 7, '0') AS STUDENT_ID,
  fn AS FIRST_NAME, ln AS LAST_NAME,
  LOWER(fn || '.' || ln || n || '@lakeside.edu') AS EMAIL,
  '555-' || LPAD(MOD(ABS(HASH(n,'ph')), 10000), 4, '0') AS PHONE,
  DATEADD(day, -(18*365 + MOD(ABS(HASH(n,'dob')), 1500)), '2026-08-24'::DATE) AS DATE_OF_BIRTH,
  college AS COLLEGE,
  CASE college
    WHEN 'Engineering'      THEN GET(ARRAY_CONSTRUCT('Mechanical Engineering','Computer Science','Electrical Engineering','Civil Engineering'), MOD(ABS(HASH(n,'maj')), 4))
    WHEN 'Arts & Sciences'  THEN GET(ARRAY_CONSTRUCT('Biology','Psychology','English','Chemistry'), MOD(ABS(HASH(n,'maj')), 4))
    WHEN 'Business'         THEN GET(ARRAY_CONSTRUCT('Accounting','Marketing','Finance','Management'), MOD(ABS(HASH(n,'maj')), 4))
    WHEN 'Nursing & Health' THEN GET(ARRAY_CONSTRUCT('Nursing','Public Health','Kinesiology','Health Sciences'), MOD(ABS(HASH(n,'maj')), 4))
    ELSE GET(ARRAY_CONSTRUCT('Elementary Education','Secondary Education','Special Education','Counseling'), MOD(ABS(HASH(n,'maj')), 4))
  END::STRING AS MAJOR,
  class_level AS CLASS_LEVEL,
  CASE class_level WHEN 'Freshman' THEN 'Fall 2026' WHEN 'Sophomore' THEN 'Fall 2025' WHEN 'Junior' THEN 'Fall 2024' ELSE 'Fall 2023' END AS ADMIT_TERM,
  CASE WHEN r > 0.97 THEN 'WITHDRAWN' WHEN r > 0.93 THEN 'LEAVE_OF_ABSENCE' ELSE 'ACTIVE' END AS ENROLLMENT_STATUS,
  IFF(MOD(ABS(HASH(n,'fg')), 100) < 32, 'Y', 'N') AS FIRST_GEN_FLAG,
  GET(ARRAY_CONSTRUCT('In-State','In-State','Out-of-State','International'), MOD(ABS(HASH(n,'res')), 4))::STRING AS RESIDENCY,
  IFF(class_level = 'Freshman', NULL,
      ROUND(GREATEST(0.8, LEAST(4.0, 3.9 - r*2.4 + (MOD(ABS(HASH(n,'gn')), 61) - 30)/100.0)), 2)) AS CUMULATIVE_GPA,
  CASE class_level WHEN 'Freshman' THEN 0 WHEN 'Sophomore' THEN 28 WHEN 'Junior' THEN 60 ELSE 92 END
    + MOD(ABS(HASH(n,'cr')), 9) AS CREDITS_EARNED
FROM b;
SELECT COUNT(*) AS STUDENTS, COUNT_IF(ENROLLMENT_STATUS<>'ACTIVE') AS NOT_ACTIVE FROM RAW.SIS_STUDENT;

In [ ]:
%%sql -r load_sis_enrollment
CREATE OR REPLACE TABLE RAW.SIS_ENROLLMENT
COMMENT = 'Registrar SIS - one row per student per course per term' AS
WITH cat AS (
  SELECT column1 AS idx, column2 AS COURSE_CODE, column3 AS COURSE_TITLE, column4 AS DEPARTMENT, column5 AS CREDITS
  FROM VALUES
   (0,'MATH101','Calculus I','Mathematics',4),(1,'ENGL101','Composition','English',3),
   (2,'BIOL110','Intro Biology','Biology',4),(3,'CHEM101','General Chemistry','Chemistry',4),
   (4,'PSYC100','Intro Psychology','Psychology',3),(5,'CS110','Intro Programming','Computer Science',4),
   (6,'ECON201','Microeconomics','Economics',3),(7,'STAT200','Statistics','Mathematics',3),
   (8,'HIST150','World History','History',3),(9,'PHYS201','Physics I','Physics',4),
   (10,'ACCT210','Financial Accounting','Accounting',3),(11,'EDUC120','Foundations of Education','Education',3),
   (12,'NURS130','Health Assessment','Nursing',4),(13,'COMM101','Public Speaking','Communication',3),
   (14,'MGMT220','Principles of Management','Management',3),(15,'ENGR100','Engineering Design','Engineering',3)
),
terms AS (SELECT column1 AS TERM, column2 AS term_idx FROM VALUES ('Fall 2025',1),('Spring 2026',2),('Fall 2026',3)),
slots AS (SELECT SEQ4() AS slot FROM TABLE(GENERATOR(ROWCOUNT => 4))),
s AS (
  SELECT STUDENT_ID, ADMIT_TERM, ENROLLMENT_STATUS, TO_NUMBER(SUBSTR(STUDENT_ID, 2)) AS n,
         RAW.SIM_RISK(TO_NUMBER(SUBSTR(STUDENT_ID, 2))) AS r
  FROM RAW.SIS_STUDENT
),
x AS (
  SELECT s.STUDENT_ID, s.n, s.r, t.TERM, t.term_idx, sl.slot,
         MOD(ABS(HASH(s.n, t.TERM)) + sl.slot * 5, 16) AS cidx,
         3.8 - s.r * 2.6 + (MOD(ABS(HASH(s.n, t.TERM, sl.slot)), 161) - 80) / 100.0
           - CASE MOD(ABS(HASH(s.n, t.TERM)) + sl.slot * 5, 16) WHEN 0 THEN 0.6 WHEN 3 THEN 0.7 WHEN 9 THEN 0.6 WHEN 2 THEN 0.3 WHEN 7 THEN 0.3 ELSE 0 END AS score
  FROM s CROSS JOIN terms t CROSS JOIN slots sl
  WHERE (s.ADMIT_TERM <> 'Fall 2026' OR t.TERM = 'Fall 2026')
    AND NOT (t.TERM = 'Fall 2026' AND s.ENROLLMENT_STATUS IN ('WITHDRAWN','LEAVE_OF_ABSENCE'))
),
l AS (
  SELECT x.*, c.COURSE_CODE, c.COURSE_TITLE, c.DEPARTMENT, c.CREDITS,
    CASE WHEN score >= 3.7 THEN 'A' WHEN score >= 3.3 THEN 'B+' WHEN score >= 3.0 THEN 'B' WHEN score >= 2.7 THEN 'B-'
         WHEN score >= 2.0 THEN 'C' WHEN score >= 1.0 THEN 'D' ELSE 'F' END AS letter
  FROM x JOIN cat c ON c.idx = x.cidx
)
SELECT
  'E' || LPAD(ROW_NUMBER() OVER (ORDER BY STUDENT_ID, term_idx, slot), 8, '0') AS ENROLLMENT_ID,
  STUDENT_ID, TERM, COURSE_CODE, COURSE_TITLE, DEPARTMENT, CREDITS,
  IFF(TERM = 'Fall 2026', NULL, IFF(r > 0.7 AND MOD(ABS(HASH(n, TERM, slot, 'w')), 10) = 0, 'W', letter)) AS FINAL_GRADE,
  IFF(TERM = 'Fall 2026', letter, NULL) AS MIDTERM_GRADE,
  CASE IFF(TERM = 'Fall 2026', letter, IFF(r > 0.7 AND MOD(ABS(HASH(n, TERM, slot, 'w')), 10) = 0, 'W', letter))
    WHEN 'A' THEN 4.0 WHEN 'B+' THEN 3.3 WHEN 'B' THEN 3.0 WHEN 'B-' THEN 2.7 WHEN 'C' THEN 2.0 WHEN 'D' THEN 1.0 WHEN 'F' THEN 0.0 END AS GRADE_POINTS,
  CASE WHEN TERM = 'Fall 2026' THEN 'ENROLLED'
       WHEN r > 0.7 AND MOD(ABS(HASH(n, TERM, slot, 'w')), 10) = 0 THEN 'WITHDRAWN' ELSE 'COMPLETED' END AS ENROLLMENT_STATUS
FROM l;
SELECT TERM, COUNT(*) AS ROWS_, COUNT(DISTINCT STUDENT_ID) AS STUDENTS FROM RAW.SIS_ENROLLMENT GROUP BY TERM ORDER BY TERM;

In [ ]:
%%sql -r load_lms
CREATE OR REPLACE TABLE RAW.LMS_ACTIVITY
COMMENT = 'LMS weekly activity, Fall 2026 weeks 1-10. Key: LMS_USER_EMAIL (no student id)' AS
WITH s AS (
  SELECT STUDENT_ID, EMAIL, TO_NUMBER(SUBSTR(STUDENT_ID, 2)) AS n, RAW.SIM_RISK(TO_NUMBER(SUBSTR(STUDENT_ID, 2))) AS r
  FROM RAW.SIS_STUDENT WHERE ENROLLMENT_STATUS = 'ACTIVE'
),
w AS (SELECT SEQ4() + 1 AS week_num FROM TABLE(GENERATOR(ROWCOUNT => 10))),
b AS (
  SELECT s.n, s.r, w.week_num,
    CASE WHEN MOD(ABS(HASH(s.n,'dom')), 100) < 3 THEN REPLACE(s.EMAIL, '@lakeside.edu', '@lakeside.com') ELSE s.EMAIL END AS e,
    GREATEST(0, ROUND(16 * (1 - 0.75 * s.r) - s.r * s.r * w.week_num * 1.3 + (MOD(ABS(HASH(s.n, w.week_num, 'lg')), 7) - 3))) AS logins,
    2 + MOD(ABS(HASH(s.n, w.week_num, 'due')), 3) AS due,
    GREATEST(0, LEAST(1, 1 - s.r * 0.9 - s.r * w.week_num / 25 + (MOD(ABS(HASH(s.n, w.week_num, 'sb')), 31) - 15) / 100.0)) AS sub_rate
  FROM s CROSS JOIN w
)
SELECT
  IFF(MOD(ABS(HASH(n,'uc')), 5) = 0, UPPER(e), e) AS LMS_USER_EMAIL,
  'Fall 2026' AS TERM, week_num AS WEEK_NUM,
  logins AS LOGINS,
  logins * (15 + MOD(ABS(HASH(n, week_num, 'mn')), 31)) AS MINUTES_ACTIVE,
  due AS ASSIGNMENTS_DUE,
  LEAST(due, ROUND(due * sub_rate)) AS ASSIGNMENTS_SUBMITTED,
  ROUND(GREATEST(35, LEAST(100, 92 - r * 45 + (MOD(ABS(HASH(n, week_num, 'sc')), 17) - 8))), 1) AS AVG_SCORE_PCT,
  GREATEST(0, ROUND(3 * (1 - r) + (MOD(ABS(HASH(n, week_num, 'dp')), 3) - 1))) AS DISCUSSION_POSTS
FROM b
UNION ALL
SELECT 'test.account' || SEQ4() || '@lakeside.edu', 'Fall 2026', 1, 1, 5, 1, 1, 100, 0
FROM TABLE(GENERATOR(ROWCOUNT => 25));
SELECT COUNT(*) AS ROWS_, COUNT(DISTINCT LOWER(LMS_USER_EMAIL)) AS DISTINCT_LMS_USERS FROM RAW.LMS_ACTIVITY;

In [ ]:
%%sql -r load_finaid_bursar
CREATE OR REPLACE TABLE RAW.FINAID_AWARD
COMMENT = 'Financial Aid 2026-27. Keys: AID_ID (FA-nnnnnn) and bare STUDENT_NUMBER. ~85% of students have a record' AS
WITH s AS (
  SELECT STUDENT_ID, RESIDENCY, TO_NUMBER(SUBSTR(STUDENT_ID, 2)) AS n, RAW.SIM_RISK(TO_NUMBER(SUBSTR(STUDENT_ID, 2))) AS r
  FROM RAW.SIS_STUDENT
)
SELECT
  'FA-' || (100000 + n * 7) AS AID_ID,
  n AS STUDENT_NUMBER,
  '2026-27' AS AID_YEAR,
  IFF(RESIDENCY = 'In-State', 34000, 46000) AS COST_OF_ATTENDANCE,
  ROUND(IFF(RESIDENCY = 'In-State', 14000, 18000) * (1 - 0.6 * r) + MOD(ABS(HASH(n,'g')), 3000), -2) AS GRANTS_AND_SCHOLARSHIPS,
  ROUND(5500 * (0.4 + r) , -2) AS LOANS,
  IFF(MOD(ABS(HASH(n,'ws')), 100) < 30, 2400, 0) AS WORK_STUDY,
  ROUND(GREATEST(0, r * r * 16000 + (MOD(ABS(HASH(n,'un')), 3001) - 1500)), -1) AS UNMET_NEED,
  IFF(r > 0.25 AND MOD(ABS(HASH(n,'pell')), 100) < 45, 'Y', 'N') AS PELL_ELIGIBLE,
  CASE WHEN r > 0.93 THEN 'SUSPENDED' WHEN r > 0.8 THEN 'WARNING' ELSE 'MEETING' END AS SAP_STATUS
FROM s
WHERE MOD(ABS(HASH(n,'fa')), 100) < 85;

CREATE OR REPLACE TABLE RAW.BURSAR_ACCOUNT
COMMENT = 'Student Accounts. Key: STUDENT_ID (same format as SIS)' AS
WITH s AS (
  SELECT STUDENT_ID, TO_NUMBER(SUBSTR(STUDENT_ID, 2)) AS n, RAW.SIM_RISK(TO_NUMBER(SUBSTR(STUDENT_ID, 2))) AS r FROM RAW.SIS_STUDENT
),
b AS (
  SELECT STUDENT_ID, n, r,
    IFF(r > 0.5, ROUND((r - 0.45) * 9000 + (MOD(ABS(HASH(n,'bal')), 801) - 400), 2), 0) AS bal
  FROM s
)
SELECT
  'B' || LPAD(n, 7, '0') AS ACCOUNT_NO,
  STUDENT_ID,
  GREATEST(bal, 0) AS BALANCE_DUE,
  IFF(bal > 0, 10 + MOD(ABS(HASH(n,'dpd')), 111), 0) AS DAYS_PAST_DUE,
  IFF(MOD(ABS(HASH(n,'pp')), 100) < 25, 'Y', 'N') AS PAYMENT_PLAN_FLAG,
  CASE WHEN bal > 2500 AND MOD(ABS(HASH(n,'dpd')), 111) + 10 > 45 THEN 'REGISTRATION_HOLD'
       WHEN bal > 0 AND MOD(ABS(HASH(n,'dpd')), 111) + 10 > 30 THEN 'FINANCIAL_HOLD' ELSE 'NONE' END AS HOLD_TYPE
FROM b;
SELECT (SELECT COUNT(*) FROM RAW.FINAID_AWARD) AS FINAID_ROWS, (SELECT COUNT(*) FROM RAW.BURSAR_ACCOUNT) AS BURSAR_ROWS,
       (SELECT COUNT_IF(HOLD_TYPE <> 'NONE') FROM RAW.BURSAR_ACCOUNT) AS STUDENTS_WITH_HOLDS;

In [ ]:
%%sql -r load_crm
CREATE OR REPLACE TABLE RAW.ADVISING_CONTACT
COMMENT = 'Advising CRM contact record. Keys: CRM_CONTACT_ID (C-n) and STUDENT_REF (STU-nnnnnnn)' AS
SELECT
  'C-' || TO_NUMBER(SUBSTR(STUDENT_ID, 2)) AS CRM_CONTACT_ID,
  'STU-' || SUBSTR(STUDENT_ID, 2)          AS STUDENT_REF,
  GET(ARRAY_CONSTRUCT('Dana Whitfield','Marcus Lee','Priya Raman','Tom Alvarez','Keisha Brown','Sam Okoye','Laura Chen','Victor Ramos'),
      MOD(ABS(HASH(STUDENT_ID,'adv')), 8))::STRING AS ASSIGNED_ADVISOR
FROM RAW.SIS_STUDENT;

CREATE OR REPLACE TABLE RAW.ADVISING_NOTE
COMMENT = 'Advising CRM free-text notes. Contains PII - do NOT expose to AI without redaction' AS
WITH s AS (
  SELECT st.STUDENT_ID, st.FIRST_NAME, TO_NUMBER(SUBSTR(st.STUDENT_ID, 2)) AS n,
         RAW.SIM_RISK(TO_NUMBER(SUBSTR(st.STUDENT_ID, 2))) AS r, c.CRM_CONTACT_ID, c.ASSIGNED_ADVISOR
  FROM RAW.SIS_STUDENT st JOIN RAW.ADVISING_CONTACT c ON c.STUDENT_REF = 'STU-' || SUBSTR(st.STUDENT_ID, 2)
),
k AS (SELECT SEQ4() + 1 AS k FROM TABLE(GENERATOR(ROWCOUNT => 3))),
x AS (
  SELECT s.*, k.k,
    (s.r + (MOD(ABS(HASH(s.n, k.k, 'cn')), 41) - 20) / 100.0) > 0.62 AS concerned,
    MOD(ABS(HASH(s.n, k.k, 'tp')), 4) AS tp
  FROM s CROSS JOIN k
  WHERE MOD(ABS(HASH(s.n, k.k, 'has')), 100) < IFF(s.r > 0.6, 70, 40)
),
t AS (
  SELECT x.*,
    CASE WHEN concerned THEN GET(ARRAY_CONSTRUCT('Financial','Wellbeing','Academic','Retention'), tp)::STRING
         ELSE GET(ARRAY_CONSTRUCT('Career','Planning','Planning','Career'), tp)::STRING END AS note_type,
    CASE WHEN concerned THEN GET(ARRAY_CONSTRUCT(
      'Met with {fn} about a growing balance and a pending registration hold. Student shared that hours at their off-campus job were cut and they are unsure they can afford spring tuition. Referred to Financial Aid for a payment plan review.',
      '{fn} reported feeling overwhelmed and has been missing classes. Mentioned trouble sleeping and homesickness. Recommended campus counseling services and scheduled a follow-up check-in in two weeks.',
      'Midterm grades in two courses are below a C. {fn} says the pace of the lab sections is hard to keep up with and has not attended tutoring. Discussed a reduced course load and the withdrawal deadline.',
      '{fn} is considering transferring or stopping out after this term, citing cost and a lack of connection to other students. Advised on leave-of-absence options and asked to reconnect before registration opens.'), tp)::STRING
    ELSE GET(ARRAY_CONSTRUCT(
      '{fn} is on track and exploring summer internships in their field. Reviewed the degree audit and all milestone courses are on schedule.',
      'Routine planning meeting. {fn} is registered for next term and is interested in a research assistant position.',
      '{fn} joined a student organization and attends study groups regularly. Feeling confident about the semester.',
      'Career conversation: {fn} wants to attend the spring career fair. Updated resume together and reviewed internship deadlines.'), tp)::STRING END AS tpl
  FROM x
)
SELECT
  'N' || LPAD(ROW_NUMBER() OVER (ORDER BY STUDENT_ID, k), 7, '0') AS NOTE_ID,
  CRM_CONTACT_ID,
  DATEADD(day, MOD(ABS(HASH(n, k, 'dt')), 70), '2026-08-24'::DATE) AS NOTE_DATE,
  ASSIGNED_ADVISOR AS ADVISOR_NAME,
  note_type AS NOTE_TYPE,
  REPLACE(tpl, '{fn}', FIRST_NAME)
    || IFF(MOD(ABS(HASH(n, k, 'ph')), 100) < 25, ' Student can be reached at 555-' || LPAD(MOD(ABS(HASH(n,'ph')), 10000), 4, '0') || '.', '')
    || IFF(MOD(ABS(HASH(n, k, 'em')), 100) < 15, ' Prefers contact via personal email ' || LOWER(FIRST_NAME) || n || '@gmail.com.', '') AS NOTE_TEXT
FROM t;
SELECT COUNT(*) AS NOTES, COUNT_IF(NOTE_TEXT ILIKE '%@gmail.com%' OR NOTE_TEXT LIKE '%555-%') AS NOTES_WITH_PII FROM RAW.ADVISING_NOTE;

In [ ]:
%%sql -r load_campus
CREATE OR REPLACE TABLE RAW.ID_CARD_ROSTER
COMMENT = 'Campus card roster. Keys: CARD_ID and EMAIL (no student id)' AS
SELECT 'CARD' || LPAD(10000 + TO_NUMBER(SUBSTR(STUDENT_ID, 2)) * 3, 7, '0') AS CARD_ID,
       EMAIL, '2026-08-15'::DATE AS ISSUED_DATE
FROM RAW.SIS_STUDENT;

CREATE OR REPLACE TABLE RAW.CAMPUS_ENGAGEMENT
COMMENT = 'Campus card swipes by week, Fall 2026 weeks 1-10. Key: CARD_ID' AS
WITH s AS (
  SELECT r.CARD_ID, TO_NUMBER(SUBSTR(st.STUDENT_ID, 2)) AS n, RAW.SIM_RISK(TO_NUMBER(SUBSTR(st.STUDENT_ID, 2))) AS rk
  FROM RAW.SIS_STUDENT st JOIN RAW.ID_CARD_ROSTER r ON r.EMAIL = st.EMAIL WHERE st.ENROLLMENT_STATUS = 'ACTIVE'
),
w AS (SELECT SEQ4() + 1 AS week_num FROM TABLE(GENERATOR(ROWCOUNT => 10)))
SELECT CARD_ID, 'Fall 2026' AS TERM, week_num AS WEEK_NUM,
  GREATEST(0, ROUND(5 * (1 - rk) - rk * week_num / 6 + (MOD(ABS(HASH(n, week_num, 'lb')), 4) - 1.5))) AS LIBRARY_VISITS,
  GREATEST(0, ROUND(3 * (1 - rk) + (MOD(ABS(HASH(n, week_num, 'rc')), 4) - 1.5))) AS REC_CENTER_VISITS,
  GREATEST(0, ROUND(10 * (1 - 0.5 * rk) + (MOD(ABS(HASH(n, week_num, 'dn')), 5) - 2))) AS DINING_SWIPES,
  GREATEST(0, ROUND(1.5 * (1 - rk) + (MOD(ABS(HASH(n, week_num, 'ev')), 3) - 1))) AS EVENTS_ATTENDED
FROM s CROSS JOIN w;
SELECT COUNT(*) AS ROWS_ FROM RAW.CAMPUS_ENGAGEMENT;

In [ ]:
%%sql -r validate_raw
SELECT 'SIS_STUDENT' AS SRC, COUNT(*) AS ROWS_ FROM RAW.SIS_STUDENT
UNION ALL SELECT 'SIS_ENROLLMENT', COUNT(*) FROM RAW.SIS_ENROLLMENT
UNION ALL SELECT 'LMS_ACTIVITY', COUNT(*) FROM RAW.LMS_ACTIVITY
UNION ALL SELECT 'FINAID_AWARD', COUNT(*) FROM RAW.FINAID_AWARD
UNION ALL SELECT 'BURSAR_ACCOUNT', COUNT(*) FROM RAW.BURSAR_ACCOUNT
UNION ALL SELECT 'ADVISING_CONTACT', COUNT(*) FROM RAW.ADVISING_CONTACT
UNION ALL SELECT 'ADVISING_NOTE', COUNT(*) FROM RAW.ADVISING_NOTE
UNION ALL SELECT 'ID_CARD_ROSTER', COUNT(*) FROM RAW.ID_CARD_ROSTER
UNION ALL SELECT 'CAMPUS_ENGAGEMENT', COUNT(*) FROM RAW.CAMPUS_ENGAGEMENT;

## 3. Identity resolution
🛠️ The SIS id is the golden key. Other systems link by exact id, prefix, exact email, and a fallback on the email local part (rescues the legacy `@lakeside.com` domain).
📌 **Best practice:** deterministic matching first, keep the method, and **measure the match rate** before anything flows downstream.

In [ ]:
%%sql -r create_xwalk
CREATE OR REPLACE TABLE CURATED.STUDENT_XWALK
  COMMENT = 'Golden-key crosswalk: SIS STUDENT_ID to every other system key, with match method'
AS
WITH lms AS (SELECT DISTINCT LOWER(LMS_USER_EMAIL) AS lms_email FROM RAW.LMS_ACTIVITY)
SELECT
  s.STUDENT_ID,
  COALESCE(l1.lms_email, l2.lms_email)                        AS LMS_EMAIL,
  CASE WHEN l1.lms_email IS NOT NULL THEN 'EMAIL_EXACT'
       WHEN l2.lms_email IS NOT NULL THEN 'EMAIL_LOCALPART' END AS LMS_MATCH_METHOD,
  f.AID_ID                                                    AS AID_ID,
  c.CRM_CONTACT_ID                                            AS CRM_CONTACT_ID,
  r.CARD_ID                                                   AS CARD_ID,
  b.ACCOUNT_NO                                                AS BURSAR_ACCOUNT_NO
FROM RAW.SIS_STUDENT s
LEFT JOIN lms l1 ON l1.lms_email = LOWER(s.EMAIL)
LEFT JOIN lms l2 ON SPLIT_PART(l2.lms_email, '@', 1) = SPLIT_PART(LOWER(s.EMAIL), '@', 1)
LEFT JOIN RAW.FINAID_AWARD f      ON f.STUDENT_NUMBER = TO_NUMBER(SUBSTR(s.STUDENT_ID, 2))
LEFT JOIN RAW.ADVISING_CONTACT c  ON c.STUDENT_REF = 'STU-' || SUBSTR(s.STUDENT_ID, 2)
LEFT JOIN RAW.ID_CARD_ROSTER r    ON LOWER(r.EMAIL) = LOWER(s.EMAIL)
LEFT JOIN RAW.BURSAR_ACCOUNT b    ON b.STUDENT_ID = s.STUDENT_ID;
SELECT COUNT(*) AS STUDENTS FROM CURATED.STUDENT_XWALK;

In [ ]:
%%sql -r match_report
SELECT 'LMS'        AS SOURCE, COUNT(LMS_EMAIL)        AS MATCHED, COUNT(*) AS STUDENTS, ROUND(100 * COUNT(LMS_EMAIL) / COUNT(*), 1) AS MATCH_PCT FROM CURATED.STUDENT_XWALK
UNION ALL SELECT 'FINANCIAL AID', COUNT(AID_ID), COUNT(*), ROUND(100 * COUNT(AID_ID) / COUNT(*), 1) FROM CURATED.STUDENT_XWALK
UNION ALL SELECT 'ADVISING CRM',  COUNT(CRM_CONTACT_ID), COUNT(*), ROUND(100 * COUNT(CRM_CONTACT_ID) / COUNT(*), 1) FROM CURATED.STUDENT_XWALK
UNION ALL SELECT 'CAMPUS CARD',   COUNT(CARD_ID), COUNT(*), ROUND(100 * COUNT(CARD_ID) / COUNT(*), 1) FROM CURATED.STUDENT_XWALK
UNION ALL SELECT 'BURSAR',        COUNT(BURSAR_ACCOUNT_NO), COUNT(*), ROUND(100 * COUNT(BURSAR_ACCOUNT_NO) / COUNT(*), 1) FROM CURATED.STUDENT_XWALK;

## 4. Feature tables as dynamic tables
🛠️ Declare the result; Snowflake keeps it fresh. We set `REFRESH_MODE = INCREMENTAL` so only changed data is reprocessed.

In [ ]:
%%sql -r create_academic
CREATE OR REPLACE DYNAMIC TABLE CURATED.FEAT_ACADEMIC
  TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL WAREHOUSE = STUDENT360_LAB_WH
  COMMENT = 'Per-student academic features from SIS enrollments'
AS
SELECT
  STUDENT_ID,
  ROUND(SUM(IFF(ENROLLMENT_STATUS = 'COMPLETED', GRADE_POINTS * CREDITS, NULL))
        / NULLIF(SUM(IFF(ENROLLMENT_STATUS = 'COMPLETED', CREDITS, NULL)), 0), 2)    AS PRIOR_TERMS_GPA,
  COUNT_IF(TERM = 'Fall 2026')                                                      AS CURRENT_COURSES,
  SUM(IFF(TERM = 'Fall 2026', CREDITS, 0))                                           AS CURRENT_CREDITS,
  ROUND(SUM(IFF(TERM = 'Fall 2026', GRADE_POINTS * CREDITS, NULL))
        / NULLIF(SUM(IFF(TERM = 'Fall 2026', CREDITS, NULL)), 0), 2)                 AS MIDTERM_GPA,
  COUNT_IF(TERM = 'Fall 2026' AND MIDTERM_GRADE IN ('D','F'))                        AS MIDTERM_DF_COUNT,
  COUNT_IF(ENROLLMENT_STATUS = 'WITHDRAWN')                                          AS COURSE_WITHDRAWALS
FROM RAW.SIS_ENROLLMENT
GROUP BY STUDENT_ID;
SELECT COUNT(*) AS ROWS_ FROM CURATED.FEAT_ACADEMIC;

In [ ]:
%%sql -r create_lms_feat
CREATE OR REPLACE DYNAMIC TABLE CURATED.FEAT_LMS
  TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL WAREHOUSE = STUDENT360_LAB_WH
  COMMENT = 'Per-student LMS engagement features (Fall 2026), resolved through the crosswalk'
AS
SELECT
  x.STUDENT_ID,
  ROUND(AVG(IFF(a.WEEK_NUM <= 3, a.LOGINS, NULL)), 1)  AS AVG_LOGINS_EARLY,
  ROUND(AVG(IFF(a.WEEK_NUM >= 8, a.LOGINS, NULL)), 1)  AS AVG_LOGINS_RECENT,
  ROUND((AVG(IFF(a.WEEK_NUM >= 8, a.LOGINS, NULL)) - AVG(IFF(a.WEEK_NUM <= 3, a.LOGINS, NULL)))
        / NULLIF(AVG(IFF(a.WEEK_NUM <= 3, a.LOGINS, NULL)), 0), 3) AS LOGIN_TREND_PCT,
  ROUND(SUM(a.ASSIGNMENTS_SUBMITTED) / NULLIF(SUM(a.ASSIGNMENTS_DUE), 0), 3)  AS ASSIGNMENT_SUBMISSION_RATE,
  ROUND(AVG(a.AVG_SCORE_PCT), 1)                       AS AVG_LMS_SCORE_PCT
FROM RAW.LMS_ACTIVITY a
JOIN CURATED.STUDENT_XWALK x ON x.LMS_EMAIL = LOWER(a.LMS_USER_EMAIL)
GROUP BY x.STUDENT_ID;
SELECT COUNT(*) AS ROWS_ FROM CURATED.FEAT_LMS;

In [ ]:
%%sql -r create_fin_feat
CREATE OR REPLACE DYNAMIC TABLE CURATED.FEAT_FINANCIAL
  TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL WAREHOUSE = STUDENT360_LAB_WH
  COMMENT = 'Per-student financial features from Financial Aid and Bursar'
AS
SELECT
  x.STUDENT_ID,
  f.COST_OF_ATTENDANCE, f.GRANTS_AND_SCHOLARSHIPS, f.LOANS, f.UNMET_NEED, f.PELL_ELIGIBLE, f.SAP_STATUS,
  b.BALANCE_DUE, b.DAYS_PAST_DUE, b.HOLD_TYPE, b.PAYMENT_PLAN_FLAG
FROM CURATED.STUDENT_XWALK x
LEFT JOIN RAW.FINAID_AWARD f ON f.AID_ID = x.AID_ID
LEFT JOIN RAW.BURSAR_ACCOUNT b ON b.ACCOUNT_NO = x.BURSAR_ACCOUNT_NO;
SELECT COUNT(*) AS ROWS_ FROM CURATED.FEAT_FINANCIAL;

In [ ]:
%%sql -r create_campus_feat
CREATE OR REPLACE DYNAMIC TABLE CURATED.FEAT_CAMPUS
  TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL WAREHOUSE = STUDENT360_LAB_WH
  COMMENT = 'Per-student campus engagement from ID-card swipes (Fall 2026)'
AS
SELECT
  x.STUDENT_ID,
  ROUND(AVG(c.LIBRARY_VISITS), 1)    AS AVG_WEEKLY_LIBRARY_VISITS,
  ROUND(AVG(c.REC_CENTER_VISITS), 1) AS AVG_WEEKLY_REC_VISITS,
  ROUND(AVG(c.DINING_SWIPES), 1)     AS AVG_WEEKLY_DINING_SWIPES,
  SUM(c.EVENTS_ATTENDED)             AS EVENTS_ATTENDED_TERM,
  ROUND(AVG(c.LIBRARY_VISITS + c.REC_CENTER_VISITS + c.EVENTS_ATTENDED), 1) AS CAMPUS_TOUCHPOINTS_PER_WEEK
FROM RAW.CAMPUS_ENGAGEMENT c
JOIN CURATED.STUDENT_XWALK x ON x.CARD_ID = c.CARD_ID
GROUP BY x.STUDENT_ID;
SELECT COUNT(*) AS ROWS_ FROM CURATED.FEAT_CAMPUS;

## 5. Unstructured advising notes with Cortex AI
🛠️ `AI_REDACT` strips names, phones and emails; `AI_CLASSIFY` tags a theme. Only the redacted text flows on.
📌 **Best practice:** redact in the data layer, not in a prompt. A prompt asks; the data layer guarantees.

In [ ]:
%%sql -r create_notes_enriched
CREATE OR REPLACE TABLE CURATED.ADVISING_NOTE_ENRICHED
COMMENT = 'Advising notes resolved to STUDENT_ID, PII-redacted (AI_REDACT) and theme-tagged (AI_CLASSIFY). Raw text is NOT carried forward.'
AS
SELECT
  n.NOTE_ID,
  x.STUDENT_ID,
  s.COLLEGE,
  n.NOTE_DATE,
  n.ADVISOR_NAME,
  n.NOTE_TYPE,
  AI_REDACT(n.NOTE_TEXT)::STRING AS NOTE_TEXT_REDACTED,
  AI_CLASSIFY(n.NOTE_TEXT,
    ['financial hardship', 'mental health and wellbeing', 'academic struggle', 'considering leaving', 'on track']):labels[0]::STRING AS NOTE_THEME
FROM RAW.ADVISING_NOTE n
JOIN CURATED.STUDENT_XWALK x ON x.CRM_CONTACT_ID = n.CRM_CONTACT_ID
JOIN RAW.SIS_STUDENT s ON s.STUDENT_ID = x.STUDENT_ID;
ALTER TABLE CURATED.ADVISING_NOTE_ENRICHED ADD COLUMN CONCERN_FLAG BOOLEAN;
UPDATE CURATED.ADVISING_NOTE_ENRICHED SET CONCERN_FLAG = (NOTE_THEME <> 'on track');
SELECT NOTE_THEME, COUNT(*) AS NOTES FROM CURATED.ADVISING_NOTE_ENRICHED GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%%sql -r check_redaction
-- Raw notes with PII vs. redacted notes still containing a phone number or gmail address (expect 0)
SELECT
  (SELECT COUNT(*) FROM RAW.ADVISING_NOTE WHERE NOTE_TEXT LIKE '%555-%' OR NOTE_TEXT ILIKE '%@gmail.com%') AS RAW_NOTES_WITH_PII,
  (SELECT COUNT(*) FROM CURATED.ADVISING_NOTE_ENRICHED WHERE NOTE_TEXT_REDACTED LIKE '%555-%' OR NOTE_TEXT_REDACTED ILIKE '%@gmail.com%') AS REDACTED_NOTES_WITH_PII;

-- Side-by-side example
SELECT r.NOTE_TEXT AS RAW_TEXT, e.NOTE_TEXT_REDACTED
FROM RAW.ADVISING_NOTE r JOIN CURATED.ADVISING_NOTE_ENRICHED e USING (NOTE_ID)
WHERE r.NOTE_TEXT LIKE '%555-%' LIMIT 2;

In [ ]:
%%sql -r create_adv_feat
CREATE OR REPLACE DYNAMIC TABLE CURATED.FEAT_ADVISING
  TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL WAREHOUSE = STUDENT360_LAB_WH
  COMMENT = 'Per-student advising features derived from enriched notes'
AS
SELECT
  STUDENT_ID,
  COUNT(*)                                     AS ADVISING_NOTES_TERM,
  COUNT_IF(CONCERN_FLAG)                       AS CONCERN_NOTES,
  MAX(NOTE_DATE)                               AS LAST_ADVISING_DATE,
  DATEDIFF('day', MAX(NOTE_DATE), '2026-11-02'::DATE) AS DAYS_SINCE_ADVISING,
  MAX_BY(IFF(CONCERN_FLAG, NOTE_THEME, NULL), IFF(CONCERN_FLAG, NOTE_DATE, NULL)) AS LATEST_CONCERN_THEME
FROM CURATED.ADVISING_NOTE_ENRICHED
GROUP BY STUDENT_ID;
SELECT COUNT(*) AS ROWS_ FROM CURATED.FEAT_ADVISING;

## 6. Assemble the Student 360
🛠️ One row per student, an explainable risk score and the **drivers** behind it. Every join is a LEFT JOIN from the SIS roster, so no student disappears.

In [ ]:
%%sql -r create_student360
CREATE OR REPLACE DYNAMIC TABLE CURATED.STUDENT_360
  TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL WAREHOUSE = STUDENT360_LAB_WH
  COMMENT = 'Student 360: one row per student unifying SIS, LMS, Financial Aid, Bursar, Advising CRM and Campus Card, with an explainable retention-risk score. As of week 10, Fall 2026.'
AS
WITH base AS (
  SELECT
    s.STUDENT_ID, s.FIRST_NAME, s.LAST_NAME, s.FIRST_NAME || ' ' || s.LAST_NAME AS FULL_NAME,
    s.EMAIL, s.PHONE, s.DATE_OF_BIRTH,
    s.COLLEGE, s.MAJOR, s.CLASS_LEVEL, s.ADMIT_TERM, s.ENROLLMENT_STATUS, s.FIRST_GEN_FLAG, s.RESIDENCY,
    COALESCE(s.CUMULATIVE_GPA, ac.MIDTERM_GPA)  AS CURRENT_GPA,
    s.CUMULATIVE_GPA, s.CREDITS_EARNED,
    ac.MIDTERM_GPA, ac.CURRENT_COURSES, ac.CURRENT_CREDITS, ac.MIDTERM_DF_COUNT, ac.COURSE_WITHDRAWALS,
    l.AVG_LOGINS_EARLY, l.AVG_LOGINS_RECENT, l.LOGIN_TREND_PCT, l.ASSIGNMENT_SUBMISSION_RATE, l.AVG_LMS_SCORE_PCT,
    f.COST_OF_ATTENDANCE, f.UNMET_NEED, f.PELL_ELIGIBLE, f.SAP_STATUS,
    f.BALANCE_DUE, f.DAYS_PAST_DUE, COALESCE(f.HOLD_TYPE, 'NONE') AS HOLD_TYPE,
    cp.CAMPUS_TOUCHPOINTS_PER_WEEK, cp.AVG_WEEKLY_LIBRARY_VISITS, cp.AVG_WEEKLY_DINING_SWIPES, cp.EVENTS_ATTENDED_TERM,
    COALESCE(av.ADVISING_NOTES_TERM, 0) AS ADVISING_NOTES_TERM, COALESCE(av.CONCERN_NOTES, 0) AS CONCERN_NOTES,
    av.LAST_ADVISING_DATE, COALESCE(av.DAYS_SINCE_ADVISING, 999) AS DAYS_SINCE_ADVISING, av.LATEST_CONCERN_THEME,
    c.ASSIGNED_ADVISOR,
    (1 + IFF(l.STUDENT_ID IS NOT NULL,1,0) + IFF(f.UNMET_NEED IS NOT NULL OR f.BALANCE_DUE IS NOT NULL,1,0)
       + IFF(cp.STUDENT_ID IS NOT NULL,1,0) + IFF(av.STUDENT_ID IS NOT NULL,1,0) + IFF(ac.STUDENT_ID IS NOT NULL,1,0)) AS DATA_SOURCES_COVERED
  FROM RAW.SIS_STUDENT s
  LEFT JOIN CURATED.FEAT_ACADEMIC  ac ON ac.STUDENT_ID = s.STUDENT_ID
  LEFT JOIN CURATED.FEAT_LMS        l ON  l.STUDENT_ID = s.STUDENT_ID
  LEFT JOIN CURATED.FEAT_FINANCIAL  f ON  f.STUDENT_ID = s.STUDENT_ID
  LEFT JOIN CURATED.FEAT_CAMPUS    cp ON cp.STUDENT_ID = s.STUDENT_ID
  LEFT JOIN CURATED.FEAT_ADVISING  av ON av.STUDENT_ID = s.STUDENT_ID
  LEFT JOIN CURATED.STUDENT_XWALK  xw ON xw.STUDENT_ID = s.STUDENT_ID
  LEFT JOIN RAW.ADVISING_CONTACT    c ON  c.CRM_CONTACT_ID = xw.CRM_CONTACT_ID
),
rules AS (
  SELECT b.*,
    IFF(CURRENT_GPA < 2.0, 25, IFF(CURRENT_GPA < 2.5, 15, IFF(CURRENT_GPA < 3.0, 5, 0)))        AS p_gpa,
    IFF(MIDTERM_DF_COUNT >= 2, 15, IFF(MIDTERM_DF_COUNT = 1, 8, 0))                              AS p_midterm,
    IFF(LOGIN_TREND_PCT <= -0.4, 15, IFF(LOGIN_TREND_PCT <= -0.2, 7, 0))                          AS p_lms_trend,
    IFF(ASSIGNMENT_SUBMISSION_RATE < 0.6, 10, IFF(ASSIGNMENT_SUBMISSION_RATE < 0.8, 4, 0))        AS p_submit,
    IFF(CAMPUS_TOUCHPOINTS_PER_WEEK < 2.5, 5, 0)                                                  AS p_campus,
    IFF(UNMET_NEED >= 8000, 10, IFF(UNMET_NEED >= 4000, 5, 0))                                    AS p_unmet,
    IFF(HOLD_TYPE <> 'NONE', 10, 0)                                                               AS p_hold,
    IFF(CONCERN_NOTES >= 2, 10, IFF(CONCERN_NOTES = 1, 6, 0))                                     AS p_concern,
    IFF(DAYS_SINCE_ADVISING > 45, 5, 0)                                                           AS p_no_contact
  FROM base b
)
SELECT
  STUDENT_ID, FIRST_NAME, LAST_NAME, FULL_NAME, EMAIL, PHONE, DATE_OF_BIRTH,
  COLLEGE, MAJOR, CLASS_LEVEL, ADMIT_TERM, ENROLLMENT_STATUS, FIRST_GEN_FLAG, RESIDENCY, ASSIGNED_ADVISOR,
  CURRENT_GPA, CUMULATIVE_GPA, CREDITS_EARNED, MIDTERM_GPA, CURRENT_COURSES, CURRENT_CREDITS, MIDTERM_DF_COUNT, COURSE_WITHDRAWALS,
  AVG_LOGINS_EARLY, AVG_LOGINS_RECENT, LOGIN_TREND_PCT, ASSIGNMENT_SUBMISSION_RATE, AVG_LMS_SCORE_PCT,
  COST_OF_ATTENDANCE, UNMET_NEED, PELL_ELIGIBLE, SAP_STATUS, BALANCE_DUE, DAYS_PAST_DUE, HOLD_TYPE,
  CAMPUS_TOUCHPOINTS_PER_WEEK, AVG_WEEKLY_LIBRARY_VISITS, AVG_WEEKLY_DINING_SWIPES, EVENTS_ATTENDED_TERM,
  ADVISING_NOTES_TERM, CONCERN_NOTES, LAST_ADVISING_DATE, DAYS_SINCE_ADVISING, LATEST_CONCERN_THEME,
  DATA_SOURCES_COVERED,
  LEAST(100, p_gpa + p_midterm + p_lms_trend + p_submit + p_campus + p_unmet + p_hold + p_concern + p_no_contact) AS RETENTION_RISK_SCORE,
  CASE WHEN LEAST(100, p_gpa + p_midterm + p_lms_trend + p_submit + p_campus + p_unmet + p_hold + p_concern + p_no_contact) >= 60 THEN 'HIGH'
       WHEN LEAST(100, p_gpa + p_midterm + p_lms_trend + p_submit + p_campus + p_unmet + p_hold + p_concern + p_no_contact) >= 30 THEN 'MEDIUM'
       ELSE 'LOW' END AS RISK_TIER,
  ARRAY_TO_STRING(ARRAY_COMPACT(ARRAY_CONSTRUCT(
    IFF(p_gpa >= 15,      'Low GPA', NULL),
    IFF(p_midterm >= 8,   'Failing/near-failing midterm grades', NULL),
    IFF(p_lms_trend >= 7, 'Declining LMS logins', NULL),
    IFF(p_submit >= 4,    'Missing assignments', NULL),
    IFF(p_campus > 0,     'Low campus engagement', NULL),
    IFF(p_unmet >= 5,     'Unmet financial need', NULL),
    IFF(p_hold > 0,       'Account hold', NULL),
    IFF(p_concern > 0,    'Advisor-noted concern', NULL),
    IFF(p_no_contact > 0, 'No recent advising contact', NULL))), '; ') AS TOP_RISK_DRIVERS
FROM rules
WHERE ENROLLMENT_STATUS = 'ACTIVE';
SELECT COUNT(*) AS STUDENTS FROM CURATED.STUDENT_360;

In [ ]:
%%sql -r check_360
-- 1) Risk distribution
SELECT RISK_TIER, COUNT(*) AS STUDENTS, ROUND(AVG(RETENTION_RISK_SCORE), 1) AS AVG_SCORE, ROUND(AVG(CURRENT_GPA), 2) AS AVG_GPA
FROM CURATED.STUDENT_360 GROUP BY 1 ORDER BY 3 DESC;

-- 2) Source coverage (how complete is each student's picture?)
SELECT DATA_SOURCES_COVERED, COUNT(*) AS STUDENTS FROM CURATED.STUDENT_360 GROUP BY 1 ORDER BY 1 DESC;

-- 3) A high-risk student, end to end
SELECT STUDENT_ID, FULL_NAME, COLLEGE, MAJOR, CURRENT_GPA, LOGIN_TREND_PCT, UNMET_NEED, HOLD_TYPE, CONCERN_NOTES,
       RETENTION_RISK_SCORE, RISK_TIER, TOP_RISK_DRIVERS
FROM CURATED.STUDENT_360 WHERE RISK_TIER = 'HIGH' ORDER BY RETENTION_RISK_SCORE DESC LIMIT 5;

## ✅ Part 1 complete
You built an identity crosswalk, five incremental feature tables, PII-safe advising notes and `CURATED.STUDENT_360`.

**Takeaways:** resolve identity deterministically and measure it; declarative pipelines (dynamic tables) beat orchestration code; redact PII before AI can read it; make scores explainable.

**Next:** Part 2 gives the data meaning and governs access.